# Lab 6: Visualizing the training of neural networks. Handwritten digit classification.

In this lab, you will learn how to use [wandb](https://wandb.ai/) to visualize the training process of neural networks. We are going to build and train a feed-forward neural network for recognizing handwritten digits of the MNIST dataset. The training process will be visualized in the wandb dashboard, which will allow us to monitor the loss and accuracy of the model in real-time.

---

Feel free to create an account at [wandb.ai](https://wandb.ai/) before starting this lab.

### A simple example of how to use wandb in a typical training loop is shown below:

```python
import wandb

wandb.login() # Log in to your wandb account

some_config = {
    'learning_rate': 0.01,
    'layer_1_size': 128,
    'layer_2_size': 64,
    'batch_size': 32
} # This is just an example of a configuration
# dictionary, you can put anything you want here

# start a new run and log parameters
wandb.init(project='mnist-classifier', config=some_config)

# Here you would prepare your data,
# initialize the model, optimizer, etc.

# Training loop
for epoch in range(100):
    ...
    wandb.log({'loss': loss, 'accuracy': accuracy})
    # This will send the loss and accuracy to wandb
    # and you can visualize it in the dashboard

# End of the run
wandb.finish()
```

The most important part is the `wandb.log()` function, which sends the data to the wandb dashboard. You can log any metric you want, not just loss and accuracy. The value passed to the function must be a dictionary.


## Exercise 1: Prepare data for training a mnist classifier (4 points)

Before you start training a neural network, you need to prepare the data. In this exercise, you will prepare the MNIST dataset of handwritten digits for training a classifier. Load the MNIST dataset using from `data/mnist_train.csv` and `data/mnist_test.csv` files. You should then:

1. Normalize the features to the range [0, 1].
2. Encode the labels using one-hot encoding.
3. Create a PyTorch `Dataset` object for the training and test sets.
4. Create a PyTorch `DataLoader` object for the training and test sets.

In [4]:
import torch
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

train = pd.read_csv('data/mnist_train.csv')
test = pd.read_csv('data/mnist_test.csv')

x_train = train.drop('label', axis=1).values.astype('float32')
y_train = train['label'].values.astype('int64')

x_test = test.drop('label', axis=1).values.astype('float32')
y_test = test['label'].values.astype('int64')

# Normalize features
x_train /= 255.0 
x_test /= 255.0

# Encode labels
x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long) 
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
y_train_onehot = F.one_hot(y_train_tensor, num_classes=10).float()
y_test_onehot = F.one_hot(y_test_tensor, num_classes=10).float()

# Create Dataset

class MNISTDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = MNISTDataset(x_train_tensor, y_train_onehot)
test_dataset = MNISTDataset(x_test_tensor, y_test_onehot)

# Create DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Exercise 2: Prepare the architecture of the neural network (4 points)

In this exercise, you will prepare the architecture of the neural network. You should:

1. Create a neural network class that inherits from `torch.nn.Module`.
2. The neural network should have at least one hidden layer.
3. Use ReLU activation functions after each but the output layer.
4. Use a softmax activation function in the output layer to get the probabilities of each class.

**Feel free to experiment with the architecture of your network** - try adding more hidden layers, changing the number of neurons in each layer, etc. You can also add a dropout layer or some other regularization technique and see if it improves the performance of your model.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Network with 1 hidden layer
class NeuralNetwork1(nn.Module):
    def __init__(self, h1_size = 392):
        super(NeuralNetwork1, self).__init__()
        
        self.l1 = nn.Linear(784, h1_size)
        self.l2 = nn.Linear(h1_size, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h1 = F.relu(self.l1(x))
        y = F.log_softmax(self.l2(h1), dim = 1)
        return y

# Network with 2 hidden layers
class NeuralNetwork2(nn.Module):
    def __init__(self, h1_size = 128, h2_size = 64):
        super(NeuralNetwork2, self).__init__()
        
        self.l1 = nn.Linear(784, h1_size)
        self.l2 = nn.Linear(h1_size, h2_size)
        self.l3 = nn.Linear(h2_size, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h1 = F.relu(self.l1(x))
        h2 = F.relu(self.l2(h1))
        y = F.log_softmax(self.l3(h2), dim = 1)
        return y

# Network with 3 hidden layers
class NeuralNetwork3(nn.Module):
    def __init__(self, h1_size = 588, h2_size = 392, h3_size = 196):
        super(NeuralNetwork3, self).__init__()

        self.l1 = nn.Linear(784, h1_size)
        self.l2 = nn.Linear(h1_size, h2_size)
        self.l3 = nn.Linear(h2_size, h3_size)
        self.l4 = nn.Linear(h3_size, 10)

    def forward(self, x):
        x = x.view(x.size(), -1)
        h1 = F.relu(self.l1(x))
        h2 = F.relu(self.l2(h1))
        h3 = F.relu(self.l3(h2))
        y = F.log_softmax(self.l4(h3), dim = 1)
        return y

# Network with 2 hidden layers and dropout
class NeuralNetwork_dropout(nn.Module):
    def __init__(self, h1_size = 128, h2_size = 64):
        super(NeuralNetwork_dropout, self).__init__()
        
        self.l1 = nn.Linear(784, h1_size)
        self.l2 = nn.Linear(h1_size, h2_size)
        self.l3 = nn.Linear(h2_size, 10)
        self.dropout = nn.Dropout(p = 0.3)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        h1 = self.dropout(F.relu(self.l1(x)))
        h2 = self.dropout(F.relu(self.l2(h1)))
        y = F.log_softmax(self.l3(h2), dim = 1)
        return y

## *Training PyTorch models on GPU

**GPUs are optimized for performing matrix operations in parallel.** Although we call them "graphics processing units", they are actually very powerful processors that can be used for any kind of parallel computation, including training deep neural networks. In fact, data science is one of the most common applications of GPUs today, as can be seen by the revenue of companies like NVIDIA over the past few years. NVIDIA is a monopolist in the GPU market - in 2023, the company owned 92% of the data center GPU market share. As for 31 July, the 2024 revenue of NVIDIA was 60.92 billion USD, while the total revenue of 2020 was $10.92 billion. If someone benefits from the deep learning hype, it is certainly NVIDIA.

If you happen to have an NVIDIA GPU in your computer, you can use it to train your deep learning models, as PyTorch has support for CUDA, which is NVIDIA's parallel computing API. To train a model on GPU, you need to explicitly tell PyTorch to move the model and the data to the GPU.

Here is an example training loop that uses the GPU:

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Check if a GPU is available

# Initialize the model and move it to the GPU
model = SomeNeuralNetwork().to(device)   # Move the model to the GPU

for epoch in range(100):
    for batch in data_loader:
        X, y = batch
        X, y = X.to(device), y.to(device)   # Move the tensors to GPU
        
        y_pred = model(X)   # Perform a forward pass (on the GPU)
        loss = criterion(y_pred, y)   # Compute the loss (still on the GPU)
        
        ...  # The rest of the training loop
        
        y_pred = y_pred.detach().cpu()   # Move the predictions back to the CPU
        # to do anything else with them
```

Note that **the model and all the tensors it uses for computation should be moved to the GPU**. You can do this by calling the `.to(device)` method on the model and the data tensors. If you want to move the data back to the CPU (to process it further, calculate metrics, visualize), you call the `.cpu()` method on the tensor.

**Doing calculations on the GPU, you should be wary of few things:**

* **The GPU has a limited amount of memory**, so you should be careful not to run out of memory. A typical graphics card has a few gigabytes of memory, so you should be fine with most models and datasets. However, moving very large tensors to the GPU can cause out-of-memory errors. That's one of the reasons why we use a dataloader and process the data in batches.
* While the GPU is much faster than the CPU for large matrix operations, **transferring data between the CPU and the GPU is slow**. Therefore, it is best to minimize the number of data transfers between the CPU and the GPU.

## Exercise 3: Prepare the training loop (2 points)

In this exercise, you will prepare the training loop. You should:

1. Initialize the neural network.
2. Define the loss function (CrossEntropyLoss) and the optimizer (Adam).
3. Pass a dictionary with the configuration to wandb. This dictionary should contain all the hyperparameters of our model, including the learning rate, the size of the hidden layers, batch size, etc.
4. Train the neural network. Each epoch should consist of a training and validation phase. You should log the loss and accuracy of the training and validation sets using wandb. Track the training progress by opening your project at [wandb.ai](https://wandb.ai/) and see how cool of a tool it is!

### Saving and loading the model
As training can take some time, it is a good idea to save the model's state dictionary (its learned weights) to a file after training. You can do this with the following code:

    torch.save(model.state_dict(), 'model.pth')
    
To load the model from the file, you can use the following code:

    model.load_state_dict(torch.load('model.pth'))

In [7]:
import wandb
import torch.optim as optim

# NeuralNetwork2

# training loop
epochs = 100  # number of epochs to train the model for
model = NeuralNetwork2(h1_size=128, h2_size=64)           #TODO: initialize the neural network model

config = {
    "h1_size": 128, 
    "h2_size": 64,
    "batch_size": 64,
    "learning_rate": 0.001,
    "epochs": 100
}

wandb.init(project="mnist_classifier", config=config)

# TODO: initialize WANDB

optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"]) #TODO: initialize the optimizer
loss = nn.NLLLoss() #TODO: initialize the loss function 
# I used NLLLoss instead, so I could keep the softmax function in the network

train_loss_history = []
test_loss_history = []

for epoch in range(epochs):
    model.train()  # set the model to training mode
    train_loss = 0

    correct_train = 0
    total_train = 0

    for X_batch, y_batch in train_loader:   # load data batch-by-batch

        optimizer.zero_grad()   # clear the gradients
        y_pred = model(X_batch)  # forward pass
        targets = torch.argmax(y_batch, dim=1)
        batch_loss = loss(y_pred, targets) # compute the loss
        batch_loss.backward()   # compute the gradients
        optimizer.step()    # update the weights

        train_loss += batch_loss.item() # accumulate training loss

        _, predicted = torch.max(y_pred, 1)
        correct_train += (predicted == targets).sum().item()
        total_train += targets.size(0)

    train_loss = train_loss / len(train_loader) # compute the average loss
    train_acc = correct_train / total_train
    train_loss_history.append(train_loss)   # save the loss for plotting
    print(f'Epoch: {epoch}')
    print(f'Train loss: {train_loss}')
    print(f'Train accuracy: {train_acc}')

    model.eval()    # set the model to evaluation mode
    test_loss = 0    # this variable will accumulate the test loss
    
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:

            y_pred = model(X_batch)  # forward pass
            targets = torch.argmax(y_batch, dim=1)
            batch_loss = loss(y_pred, targets) 
            test_loss += batch_loss.item()    # accumulate test loss

            _, predicted = torch.max(y_pred, 1)
            correct_test += (predicted == targets).sum().item()
            total_test += targets.size(0)
    
    test_loss = test_loss / len(test_loader)
    test_acc = correct_test / total_test
    test_loss_history.append(test_loss)   # save the loss for plotting
    print(f'Test loss: {test_loss}')
    print(f'Test accuracy: {test_acc}')

    wandb.log({
        "epoch": epoch+1,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "train_acc": train_acc,
        "test_acc": test_acc
    })


torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))

Epoch: 0
Train loss: 0.34930790079904517
Train accuracy: 0.9030333333333334
Test loss: 0.17193121519117693
Test accuracy: 0.948
Epoch: 1
Train loss: 0.14148519593459774
Train accuracy: 0.9581166666666666
Test loss: 0.10967849370219691
Test accuracy: 0.9666
Epoch: 2
Train loss: 0.09502679926813888
Train accuracy: 0.9716333333333333
Test loss: 0.09650173453928511
Test accuracy: 0.9693
Epoch: 3
Train loss: 0.07219590061480231
Train accuracy: 0.9777833333333333
Test loss: 0.08844522139443838
Test accuracy: 0.9718
Epoch: 4
Train loss: 0.05571189311060951
Train accuracy: 0.9826666666666667
Test loss: 0.073546092042434
Test accuracy: 0.9776
Epoch: 5
Train loss: 0.043624501323142745
Train accuracy: 0.9862166666666666
Test loss: 0.0716885922678337
Test accuracy: 0.9788
Epoch: 6
Train loss: 0.036582444896122245
Train accuracy: 0.9883
Test loss: 0.08538096705888165
Test accuracy: 0.9749
Epoch: 7
Train loss: 0.030463712527941632
Train accuracy: 0.9903666666666666
Test loss: 0.08062857693548027
Tes

<All keys matched successfully>

In [8]:
# NeuralNetwork_dropout

# training loop
epochs = 100  # number of epochs to train the model for
model = NeuralNetwork_dropout(h1_size=128, h2_size=64)           #TODO: initialize the neural network model

config = {
    "h1_size": 128, 
    "h2_size": 64,
    "batch_size": 64,
    "learning_rate": 0.001,
    "epochs": 100,
    "p": 0.3
}

wandb.init(project="mnist_classifier", config=config)

# TODO: initialize WANDB

optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"]) #TODO: initialize the optimizer
loss = nn.NLLLoss() #TODO: initialize the loss function 
# I used NLLLoss instead, so I could keep the softmax function in the network

train_loss_history = []
test_loss_history = []

for epoch in range(epochs):
    model.train()  # set the model to training mode
    train_loss = 0

    correct_train = 0
    total_train = 0

    for X_batch, y_batch in train_loader:   # load data batch-by-batch

        optimizer.zero_grad()   # clear the gradients
        y_pred = model(X_batch)  # forward pass
        targets = torch.argmax(y_batch, dim=1)
        batch_loss = loss(y_pred, targets) # compute the loss
        batch_loss.backward()   # compute the gradients
        optimizer.step()    # update the weights

        train_loss += batch_loss.item() # accumulate training loss

        _, predicted = torch.max(y_pred, 1)
        correct_train += (predicted == targets).sum().item()
        total_train += targets.size(0)

    train_loss = train_loss / len(train_loader) # compute the average loss
    train_acc = correct_train / total_train
    train_loss_history.append(train_loss)   # save the loss for plotting
    print(f'Epoch: {epoch}')
    print(f'Train loss: {train_loss}')
    print(f'Train accuracy: {train_acc}')

    model.eval()    # set the model to evaluation mode
    test_loss = 0    # this variable will accumulate the test loss
    
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:

            y_pred = model(X_batch)  # forward pass
            targets = torch.argmax(y_batch, dim=1)
            batch_loss = loss(y_pred, targets) 
            test_loss += batch_loss.item()    # accumulate test loss

            _, predicted = torch.max(y_pred, 1)
            correct_test += (predicted == targets).sum().item()
            total_test += targets.size(0)
    
    test_loss = test_loss / len(test_loader)
    test_acc = correct_test / total_test
    test_loss_history.append(test_loss)   # save the loss for plotting
    print(f'Test loss: {test_loss}')
    print(f'Test accuracy: {test_acc}')

    wandb.log({
        "epoch": epoch+1,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "train_acc": train_acc,
        "test_acc": test_acc
    })


torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█
test_acc,▁▃▆▆▅▆▄▇▇▆▅█▇▇▆▇▇▆▇▇▇▇▇▇▇▇▆▇▇▇▇▇▆▅▆█▆▇▇▅
test_loss,▃▂▂▁▁▂▃▂▃▃▂▃▃▃▃▄▅▅▄▄▅▆▄▄▅▅▅▅▅▅█▆▆▆▇▆▆▇█▆
train_acc,▁▂▃▅▅▆▇▇▆▇▇▇▇▇▇▇█▇▇█▇▇▇▇█▇▇▇▇▇███▇██████
train_loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,100
test_acc,0.9777
test_loss,0.23633
train_acc,0.99835
train_loss,0.00725


Epoch: 0
Train loss: 0.46420702736959785
Train accuracy: 0.8633
Test loss: 0.1827013413881539
Test accuracy: 0.9432
Epoch: 1
Train loss: 0.22076127747260432
Train accuracy: 0.9350333333333334
Test loss: 0.12668138862050665
Test accuracy: 0.9613
Epoch: 2
Train loss: 0.17250463052758022
Train accuracy: 0.94965
Test loss: 0.10671200407528063
Test accuracy: 0.9661
Epoch: 3
Train loss: 0.14611118449879162
Train accuracy: 0.9572333333333334
Test loss: 0.10275699003139113
Test accuracy: 0.9691
Epoch: 4
Train loss: 0.12971582023629438
Train accuracy: 0.9609166666666666
Test loss: 0.09524010729443602
Test accuracy: 0.972
Epoch: 5
Train loss: 0.11762092318703561
Train accuracy: 0.96405
Test loss: 0.08165459615183226
Test accuracy: 0.9758
Epoch: 6
Train loss: 0.1108638460743926
Train accuracy: 0.9667
Test loss: 0.08438962063324641
Test accuracy: 0.9755
Epoch: 7
Train loss: 0.10140201056312555
Train accuracy: 0.9690333333333333
Test loss: 0.08318581143661052
Test accuracy: 0.9773
Epoch: 8
Train lo

<All keys matched successfully>

In [9]:
# NeuralNetwork3

# training loop
epochs = 100  # number of epochs to train the model for
model = NeuralNetwork3(h1_size=588, h2_size=392, h3_size=196)           #TODO: initialize the neural network model

config = {
    "h1_size": 588, 
    "h2_size": 392,
    "h3_size": 196,
    "batch_size": 64,
    "learning_rate": 0.001,
    "epochs": 100
}

wandb.init(project="mnist_classifier", config=config)

# TODO: initialize WANDB

optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"]) #TODO: initialize the optimizer
loss = nn.NLLLoss() #TODO: initialize the loss function 
# I used NLLLoss instead, so I could keep the softmax function in the network

train_loss_history = []
test_loss_history = []

for epoch in range(epochs):
    model.train()  # set the model to training mode
    train_loss = 0

    correct_train = 0
    total_train = 0

    for X_batch, y_batch in train_loader:   # load data batch-by-batch

        optimizer.zero_grad()   # clear the gradients
        y_pred = model(X_batch)  # forward pass
        targets = torch.argmax(y_batch, dim=1)
        batch_loss = loss(y_pred, targets) # compute the loss
        batch_loss.backward()   # compute the gradients
        optimizer.step()    # update the weights

        train_loss += batch_loss.item() # accumulate training loss

        _, predicted = torch.max(y_pred, 1)
        correct_train += (predicted == targets).sum().item()
        total_train += targets.size(0)

    train_loss = train_loss / len(train_loader) # compute the average loss
    train_acc = correct_train / total_train
    train_loss_history.append(train_loss)   # save the loss for plotting
    print(f'Epoch: {epoch}')
    print(f'Train loss: {train_loss}')
    print(f'Train accuracy: {train_acc}')

    model.eval()    # set the model to evaluation mode
    test_loss = 0    # this variable will accumulate the test loss
    
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:

            y_pred = model(X_batch)  # forward pass
            targets = torch.argmax(y_batch, dim=1)
            batch_loss = loss(y_pred, targets) 
            test_loss += batch_loss.item()    # accumulate test loss

            _, predicted = torch.max(y_pred, 1)
            correct_test += (predicted == targets).sum().item()
            total_test += targets.size(0)
    
    test_loss = test_loss / len(test_loader)
    test_acc = correct_test / total_test
    test_loss_history.append(test_loss)   # save the loss for plotting
    print(f'Test loss: {test_loss}')
    print(f'Test accuracy: {test_acc}')

    wandb.log({
        "epoch": epoch+1,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "train_acc": train_acc,
        "test_acc": test_acc
    })


torch.save(model.state_dict(), 'model.pth')
model.load_state_dict(torch.load('model.pth'))

epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
test_acc,▁▃▅▄▇▆▆▇▇▆▇▇▇▇▆▇▇▇▇▇▇▇▆██▇▇▇▇█▇▇▆▇▇█▇▇█▇
test_loss,█▄▂▁▁▂▂▂▁▂▂▂▂▃▃▂▃▃▂▃▃▄▄▄▄▃▃▅▆▄▄▄▅▆▅▆▇▇▆▇
train_acc,▁▆▇▇▇▇▇▇████████████████████████████████
train_loss,█▄▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,100
test_acc,0.9792
test_loss,0.12198
train_acc,0.9896
train_loss,0.0331


Epoch: 0
Train loss: 0.241456976252745
Train accuracy: 0.9257166666666666
Test loss: 0.1379409193151755
Test accuracy: 0.9584
Epoch: 1
Train loss: 0.09239771792879586
Train accuracy: 0.97075
Test loss: 0.08994254544169707
Test accuracy: 0.9708
Epoch: 2
Train loss: 0.06410168051714205
Train accuracy: 0.9799833333333333
Test loss: 0.08772732491681358
Test accuracy: 0.9746
Epoch: 3
Train loss: 0.04552734514009823
Train accuracy: 0.9855333333333334
Test loss: 0.07494769320676065
Test accuracy: 0.9779
Epoch: 4
Train loss: 0.03888465637690946
Train accuracy: 0.9877
Test loss: 0.08988392704106596
Test accuracy: 0.976
Epoch: 5
Train loss: 0.03249832337324037
Train accuracy: 0.9899333333333333
Test loss: 0.06947759774244147
Test accuracy: 0.9811
Epoch: 6
Train loss: 0.025706946549212698
Train accuracy: 0.9919166666666667
Test loss: 0.08225023490589951
Test accuracy: 0.9794
Epoch: 7
Train loss: 0.024517006780821356
Train accuracy: 0.9926166666666667
Test loss: 0.08757793530246559
Test accuracy: 

KeyboardInterrupt: 

## *Exercise: Easy hyperparameter tuning with wandb

Wandb allows you to perform hyperparameter tuning by automatically creating multiple runs with different hyperparameters and logging the performance of each run. Below is a brief instruction to `wandb` hyperparameter tuning, but you are more than welcome to find more information in the [official wandb guide](https://docs.wandb.ai/guides/sweeps/).

Your task is to use wandb to perform hyperparameter tuning of the neural network, trying different values of the learning rate, batch size, and the size of the hidden layers. You can use the following hyperparameters:

First, we need to define a dictionary with the hyperparameters that we want to tune. For example:

```python
parameters = {
    'learning_rate': {'values': [0.01, 0.001, 0.0001]},
    'batch_size': {'values': [32, 64, 128]},
    'layer_1_size': {'values': [64, 128, 256]},
    'layer_2_size': {'values': [32, 64, 128]}
}
```

Then we need to create a dictionary with the configuration of the run:

```python
sweep_config = {
    'name': 'mnist-sweep',
    'method': 'grid',   # grid search, you can also try 'random' or 'bayes'
    'metric': {'goal': 'minimize', 'name': 'val_loss'},
    'parameters': parameters,   # that's the dictionary with the hyperparameters
}
```

Finally, we can use the `wandb.sweep` function to perform hyperparameter tuning:

```python
sweep_id = wandb.sweep(sweep_config, project='mnist-classifier')
```

After that, we can finally run the sweep:

```python
wandb.agent(sweep_id, function=train)
```
where `train` is a function that trains the model and logs the metrics to wandb. This function should take a `config` argument, which will contain the hyperparameters of the run. That is how wandb knows which hyperparameters to tune.

1. Rewrite the training loop into a function that takes a single dictionary `parameters` as an argument, initializes the model, optimizer, and criterion, and trains the model for a fixed number of epochs. The function should log the loss and accuracy of the training and validation sets to wandb.
2. Create a dictionary with the hyperparameters that you want to tune.
3. Create a sweep configuration dictionary.
4. Run the sweep and monitor the results in the wandb dashboard.

In [ ]:
def train(parameters: dict):
    # your code goes here
    ...

In [ ]:
parameters = {...}

sweep_config = {
    'name': 'mnist-sweep',
    'method': 'bayes',
    'metric': {'goal': 'maximize', 'name': 'accuracy'}, # if we want to maximize the accuracy
    # remember to log the metric that you want to maximize or minimize!
    'parameters': parameters,
}

sweep_id = wandb.sweep(sweep_config, project='mnist-classifier')    # This will create a new sweep
wandb.agent(sweep_id, function=train)   # This will start the hyperparameter tuning process